# 02. Supervised Modeling: Modelos de clasificación para predecir `is_recommended`

## Introducción

Este notebook aborda el problema de **clasificación supervisada**, cuyo objetivo es predecir la variable `is_recommended`, la cual indica si un usuario recomienda o no un producto cosmético.

Este problema es relevante en el análisis de comportamiento de usuarios, ya que permite comprender patrones de satisfacción y preferencia dentro del catálogo de productos.

Los datos utilizados provienen de un pipeline de preprocesamiento previo, el cual incluye transformación de variables categóricas, imputación de valores faltantes, escalado y eliminación de leakage entre conjuntos de entrenamiento y prueba.

## Consideraciones del problema

- Tipo de problema: Clasificación binaria
- Variable objetivo: `is_recommended`
- Posible desbalance de clases
- Evaluación basada en F1-score debido a la necesidad de equilibrar precision y recall

## Objetivos del notebook

1. Cargar datos procesados desde `data/processed/`.
2. Analizar la distribución de la variable objetivo.
3. Establecer un baseline con `DummyClassifier`.
4. Evaluar modelos supervisados mediante validación cruzada estratificada.
5. Optimizar hiperparámetros con Optuna.
6. Seleccionar el mejor modelo basado en desempeño CV.
7. Entrenar el modelo final con todo el conjunto de entrenamiento.
8. Guardar el modelo entrenado para uso posterior.

In [1]:
# ---------------------------
# Se importan las librerías necesarias para el proyecto de clasificación supervisada
# ---------------------------

# Manipulación de datos
import pandas as pd
import numpy as np

# Carga y guardado de modelos
import joblib

# Manejo de rutas del sistema de archivos
from pathlib import Path

# Modelos base para clasificación supervisada
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Estrategia de validación cruzada estratificada
# (mantiene proporción de clases
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Librería para optimización de hiperparámetros
import optuna

# ---------------------------
# Configuración de Optuna
# ---------------------------

# Se ajusta el nivel de logs de Optuna para evitar salida excesiva en el notebook (mantener limpieza visual)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Carga de datos y análisis inicial del balance

En esta sección se cargan únicamente los datos correspondientes al conjunto de entrenamiento previamente preprocesado:

- `X_train_processed.pkl`
- `y_train.pkl`

Es importante destacar que en esta etapa **no se utiliza el conjunto de test**, ya que este se reserva exclusivamente para la evaluación final del modelo, evitando cualquier tipo de data leakage.

Posteriormente, se analiza la estructura de los datos mediante:
- Dimensión del conjunto de características (`X_train`)
- Dimensión de la variable objetivo (`y_train`)
- Distribución de clases en `y_train` (conteos y porcentajes)

Este análisis es fundamental para identificar posibles problemas de **desbalance de clases**, lo cual influye directamente en la selección de métricas de evaluación y estrategias de validación.

Dado que un desbalance puede afectar negativamente métricas como accuracy, se opta por utilizar **F1-score como métrica principal**, además de emplear **validación cruzada estratificada**, la cual preserva la proporción de clases en cada partición del entrenamiento.

In [3]:
# ---------------------------
# Carga de rutas a datos procesados
# ---------------------------

# Se define el directorio donde están los archivos procesados
BASE_DIR = Path.cwd().parent  # sube desde /notebooks
data_dir = BASE_DIR / "data" / "processed"

# Rutas específicas de los archivos de entrenamiento
X_train_path = data_dir / "X_train_processed.pkl"
y_train_path = data_dir / "y_train.pkl"

# ---------------------------
# Carga de datos en memoria
# ---------------------------

# Se cargan las características preprocesadas del conjunto de entrenamiento
X_train = joblib.load(X_train_path)

# Se carga la variable objetivo del conjunto de entrenamiento
y_train = joblib.load(y_train_path)

# ---------------------------
# Verificación de dimensiones
# ---------------------------

# Se imprime la forma del dataset de entrada
print("X_train shape:", X_train.shape)

# Se imprime la forma de la variable objetivo
# Se convierte a numpy array para asegurar consistencia en la forma
print("y_train shape:", np.asarray(y_train).shape)

# ---------------------------
# Análisis de balance de clases
# ---------------------------

# Se convierte y_train a Series para facilitar el análisis
y_train_series = pd.Series(y_train)

# Conteo de cada clase (0 = no recomendado, 1 = recomendado)
# sort_index asegura orden consistente de las clases
class_counts = y_train_series.value_counts(dropna=False).sort_index()

# Se calcula el porcentaje que representa cada clase
class_percent = (class_counts / class_counts.sum() * 100).round(2)

# ---------------------------
# Resultados del análisis
# ---------------------------

print("\nDistribución de clases en y_train (conteos):")
print(class_counts)
print("\nDistribución de clases en y_train (porcentajes):")
print(class_percent)

X_train shape: (741138, 385)
y_train shape: (741138,)

Distribución de clases en y_train (conteos):
is_recommended
0.0    118610
1.0    622528
Name: count, dtype: int64

Distribución de clases en y_train (porcentajes):
is_recommended
0.0    16.0
1.0    84.0
Name: count, dtype: float64


### Análisis del balance de clases

Con la salida anterior se observa una distribución de clases **desbalanceada**:

- Clase 0 (No recomendado): ~16%
- Clase 1 (Recomendado): ~84%

Esto indica que el dataset está sesgado hacia la clase positiva, lo cual es común en sistemas de recomendación, donde los usuarios tienden a dejar valoraciones favorables con mayor frecuencia.

### Implicancias del desbalance
Este desbalance puede afectar el rendimiento del modelo, ya que:
- Un modelo naive podría predecir mayoritariamente la clase 1 y aun así obtener alta accuracy.
- La métrica **F1-score** es más adecuada que accuracy, ya que balancea precisión y recall.
- Es importante evaluar el desempeño por clase y no solo globalmente.

### Estrategia adoptada
Para manejar este escenario, se utilizará:
- **Validación cruzada estratificada (StratifiedKFold)**, asegurando que cada fold mantenga la proporción original de clases.
- Métricas como **F1-score** como criterio principal de evaluación.
- Ajuste de `class_weight="balanced"` en algunos modelos.

## Definición del baseline

El baseline permite establecer un punto de comparación mínimo para evaluar los modelos supervisados. Su objetivo es determinar si los modelos entrenados realmente están aprendiendo patrones relevantes o solo están reproduciendo tendencias simples del dataset.

En este caso, se utiliza `DummyClassifier` con estrategia **most_frequent**, lo que significa que el modelo siempre predice la clase mayoritaria del conjunto de entrenamiento.

Dado que el dataset presenta un desbalance de clases (aproximadamente 84% clase 1 y 16% clase 0), este baseline representa un escenario muy simple pero importante para comparación.

### Evaluación del baseline
El modelo será evaluado utilizando:

- **Validación Cruzada Estratificada**: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
- **Métrica principal**: F1-score

El uso de F1-score es fundamental debido al desbalance de clases, ya que permite evaluar el equilibrio entre precisión y recall, evitando interpretaciones engañosas basadas únicamente en accuracy.

In [4]:
# -------------------------------------------------------
# Validación cruzada para baseline (DummyClassifier)
# -------------------------------------------------------

# Se define la estrategia de validación cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -------------------------------------------------------
# Definición del modelo baseline
# -------------------------------------------------------

# DummyClassifier se utiliza como modelo base de referencia.
# Estrategia "most_frequent":
# - Siempre predice la clase mayoritaria del conjunto de entrenamiento.
# - En este caso es adecuado debido al desbalance de clases (84% vs 16%).
baseline_model = DummyClassifier(strategy="most_frequent")

# -------------------------------------------------------
# Evaluación del baseline con validación cruzada
# -------------------------------------------------------

# Se evalúa el modelo utilizando F1-score como métrica principal debido al desbalance de clases
baseline_f1_scores = cross_val_score(
    estimator=baseline_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring="f1"
)

# -------------------------------------------------------
# Resultados del baseline
# -------------------------------------------------------

print("Baseline (DummyClassifier, strategy='most_frequent')")

print(f"Media F1: {baseline_f1_scores.mean():.6f}")
print(f"Desviación estándar F1: {baseline_f1_scores.std(ddof=1):.6f}")

Baseline (DummyClassifier, strategy='most_frequent')
Media F1: 0.913021
Desviación estándar F1: 0.000000


### Interpretación del baseline

El resultado del baseline (DummyClassifier) muestra un F1-score elevado (≈0.91), sin embargo esto no indica un buen desempeño del modelo.

Esto ocurre debido al fuerte desbalance de clases del dataset (84% clase positiva), lo que provoca que un modelo que siempre predice la clase mayoritaria obtenga métricas aparentemente altas.

Por esta razón, el baseline no debe interpretarse como un modelo predictivo útil, sino como una referencia mínima de comparación.

El valor real de mejora se evaluará comparando los modelos supervisados contra este punto de referencia.

## Entrenamiento y evaluación de modelos candidatos (CV 5-fold, métrica F1)

En esta etapa se evalúan distintos modelos de clasificación supervisada con el objetivo de comparar su desempeño bajo un mismo esquema de validación.

Dado el desbalance de clases presente en el dataset (aprox. 84% clase positiva y 16% clase negativa), se utiliza la métrica **F1-score**, ya que permite equilibrar precisión y recall, evitando interpretaciones sesgadas.

**Esquema de evaluación**
Todos los modelos son evaluados utilizando:

- Validación cruzada estratificada: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
- Métrica de evaluación: F1-score

El uso de validación cruzada estratificada asegura que cada partición mantenga la proporción original de clases, lo cual es fundamental en problemas desbalanceados.

**Modelos evaluados**
Se seleccionan tres modelos base ampliamente utilizados en problemas de clasificación:

1. **Regresión Logística**
   - Modelo lineal interpretable
   - Sirve como referencia de desempeño lineal

2. **Random Forest**
   - Modelo basado en ensamblado de árboles
   - Captura relaciones no lineales y es robusto a ruido

3. **Gradient Boosting**
   - Modelo secuencial de boosting
   - Suele ofrecer alto rendimiento en problemas tabulares

Todos los modelos se utilizan con hiperparámetros por defecto para establecer una comparación inicial justa.

**Objetivo de esta etapa**
- Comparar el rendimiento promedio (F1-score) de cada modelo
- Analizar la estabilidad mediante la desviación estándar
- Seleccionar el modelo base más prometedor para la optimización de hiperparámetros (Optuna)

In [5]:
# =======================================================
# CREACIÓN DE MUESTRA
# =======================================================

from sklearn.model_selection import train_test_split

# Se extrae una muestra representativa para reducir tiempos de entrenamiento y validación cruzada.

X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=100000,
    stratify=y_train,
    random_state=42
)

print("Shape muestra X:", X_train_sample.shape)
print("Shape muestra y:", y_train_sample.shape)

Shape muestra X: (100000, 385)
Shape muestra y: (100000,)


In [6]:
# -------------------------------------------------------
# Diccionario para almacenar resultados de modelos
# -------------------------------------------------------
models_results = {}

# =======================================================
# 1) REGRESIÓN LOGÍSTICA
# =======================================================

log_reg = LogisticRegression(max_iter=1000)

log_reg_scores = cross_val_score(
    estimator=log_reg,
    X=X_train_sample,
    y=y_train_sample,
    cv=cv,
    scoring="f1"
)

models_results["LogisticRegression"] = {
    "mean_f1": log_reg_scores.mean(),
    "std_f1": log_reg_scores.std(ddof=1),
}

print("\n==============================")
print("Regresión Logística")
print("==============================")
print(f"Media F1: {log_reg_scores.mean():.6f}")
print(f"Desviación estándar F1: {log_reg_scores.std(ddof=1):.6f}")


# =======================================================
# 2) RANDOM FOREST
# =======================================================

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_scores = cross_val_score(
    estimator=rf,
    X=X_train_sample,
    y=y_train_sample,
    cv=cv,
    scoring="f1"
)

models_results["RandomForestClassifier"] = {
    "mean_f1": rf_scores.mean(),
    "std_f1": rf_scores.std(ddof=1),
}

print("\n==============================")
print("Random Forest")
print("==============================")
print(f"Media F1: {rf_scores.mean():.6f}")
print(f"Desviación estándar F1: {rf_scores.std(ddof=1):.6f}")


# =======================================================
# 3) GRADIENT BOOSTING
# =======================================================

gb = GradientBoostingClassifier(
    random_state=42
)

gb_scores = cross_val_score(
    estimator=gb,
    X=X_train_sample,
    y=y_train_sample,
    cv=cv,
    scoring="f1"
)

models_results["GradientBoostingClassifier"] = {
    "mean_f1": gb_scores.mean(),
    "std_f1": gb_scores.std(ddof=1),
}

print("\n==============================")
print("Gradient Boosting")
print("==============================")
print(f"Media F1: {gb_scores.mean():.6f}")
print(f"Desviación estándar F1: {gb_scores.std(ddof=1):.6f}")


Regresión Logística
Media F1: 0.976834
Desviación estándar F1: 0.000791

Random Forest
Media F1: 0.974725
Desviación estándar F1: 0.000636

Gradient Boosting
Media F1: 0.977453
Desviación estándar F1: 0.001022


### Interpretación de resultados

Los tres modelos evaluados (Regresión Logística, Random Forest y Gradient Boosting) presentan un desempeño muy similar en términos de F1-score, con valores cercanos a 0.97.

El modelo con mejor rendimiento promedio fue Gradient Boosting (F1 = 0.9775), seguido muy de cerca por Regresión Logística (F1 = 0.9768) y Random Forest (F1 = 0.9747).

Las diferencias observadas entre modelos son pequeñas, lo que sugiere que el problema de clasificación es altamente predecible utilizando las variables disponibles. Esto indica que las características del dataset contienen información relevante para explicar la variable objetivo `is_recommended`.

Asimismo, las desviaciones estándar obtenidas son bajas en todos los casos, lo que evidencia un comportamiento estable entre los distintos folds de validación cruzada y una buena capacidad de generalización sobre datos no observados durante cada entrenamiento.

Debido a que los resultados son cercanos entre sí, se realizará una etapa adicional de optimización de hiperparámetros mediante Optuna sobre RandomForestClassifier, con el objetivo de evaluar si es posible obtener mejoras adicionales en el desempeño predictivo.

## Optimización de hiperparámetros con Optuna (RandomForestClassifier)

Se selecciona `RandomForestClassifier` para optimización, utilizando como objetivo maximizar el **F1 promedio** en validación cruzada.

**Búsqueda de hiperparámetros**
- `n_estimators`: int [50, 300]
- `max_depth`: int [5, 20] o `None`
- `min_samples_split`: int [2, 10]
- `min_samples_leaf`: int [1, 10]

**Estrategia de evaluación**
Dentro de la función `objective(trial)`:
- Entrenar/validar con `cross_val_score` bajo:
  - `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
  - scoring = `f1`
- Retornar el F1 promedio.

Se ejecutará un estudio con al menos 20 trials.


In [7]:
# =========================================================
# FUNCIÓN OBJETIVO PARA OPTUNA
# =========================================================
# Esta función define qué se va a optimizar.
# En este caso: maximizar el F1-score promedio en CV
# para un RandomForestClassifier.
def objective(trial: optuna.trial.Trial) -> float:

    # -----------------------------------------------------
    # 1. DEFINICIÓN DEL ESPACIO DE HIPERPARÁMETROS
    # -----------------------------------------------------

    # Número de árboles en el bosque
    n_estimators = trial.suggest_int("n_estimators", 50, 300)

    # Cantidad mínima de muestras para dividir un nodo
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    # Cantidad mínima de muestras en una hoja
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)

    # -----------------------------------------------------
    # 2. CONTROL DE max_depth (ENTERO O NONE)
    # -----------------------------------------------------
    # Se permite que el modelo tenga profundidad ilimitada (None)
    # o una profundidad acotada entre 5 y 20
    use_none = trial.suggest_categorical("max_depth_is_none", [True, False])

    if use_none:
        max_depth = None
    else:
        max_depth = trial.suggest_int("max_depth", 5, 20)

    # -----------------------------------------------------
    # 3. CREACIÓN DEL MODELO
    # -----------------------------------------------------
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1  # paralelización para acelerar entrenamiento
    )

    # -----------------------------------------------------
    # 4. VALIDACIÓN CRUZADA
    # -----------------------------------------------------
    # Se evalúa el modelo con StratifiedKFold (definido antes)
    # usando F1 como métrica principal
    scores = cross_val_score(
        estimator=model,
        X=X_train_sample,
        y=y_train_sample,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )

    # -----------------------------------------------------
    # 5. RETORNO DEL RESULTADO
    # -----------------------------------------------------
    # Optuna intentará maximizar este valor
    return float(np.mean(scores))


# =========================================================
# EJECUCIÓN DEL ESTUDIO OPTUNA
# =========================================================

# Se crea el estudio de optimización
# direction="maximize" porque queremos maximizar F1
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

# Se ejecutan 20 pruebas (trials)
study.optimize(objective, n_trials=10)

# ---------------------------------------------------------
# RESULTADOS FINALES
# ---------------------------------------------------------
print("\nOptuna terminado.")
print(f"Mejor F1 CV (promedio): {study.best_value:.6f}")
print("Mejores hiperparámetros encontrados:")
print(study.best_params)


Optuna terminado.
Mejor F1 CV (promedio): 0.975970
Mejores hiperparámetros encontrados:
{'n_estimators': 179, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_depth_is_none': True}


### Interpretación de la optimización con Optuna

Se realizó una búsqueda de hiperparámetros utilizando Optuna sobre un RandomForestClassifier, empleando validación cruzada estratificada y la métrica F1 como criterio de optimización.

El mejor modelo encontrado obtuvo un F1 promedio de 0.975970, utilizando los siguientes hiperparámetros:

- n_estimators = 179
- min_samples_split = 7
- min_samples_leaf = 1
- max_depth = None

Los resultados muestran una mejora marginal respecto al Random Forest con hiperparámetros por defecto. Esto sugiere que el modelo ya presentaba un desempeño elevado antes de la optimización y que existe un límite práctico en la mejora alcanzable con ajustes de hiperparámetros.

Además, el desempeño optimizado continúa siendo similar al obtenido por Regresión Logística y Gradient Boosting, confirmando que el problema presenta una alta capacidad predictiva independientemente del algoritmo utilizado.

## Interpretación y selección final

Se compararán los resultados F1 (media y desviación estándar) obtenidos para:
- Baseline `DummyClassifier(strategy="prior")`
- `LogisticRegression(max_iter=1000)`
- `RandomForestClassifier` (base)
- `GradientBoostingClassifier` (base)
- `RandomForestClassifier` (optimizado con Optuna)

Luego se seleccionará el mejor modelo según el F1 promedio en validación cruzada.


In [8]:
# Consolidar baseline en el diccionario
models_results["DummyClassifier (prior baseline)"] = {
    "mean_f1": baseline_f1_scores.mean(),
    "std_f1": baseline_f1_scores.std(ddof=1),
}

# Convertir estudio optuna a hiperparámetros aplicables para max_depth
best_params = dict(study.best_params)

max_depth_is_none = best_params.pop("max_depth_is_none", False)
if max_depth_is_none:
    best_params["max_depth"] = None
else:
    # Si no es None, Optuna guarda el valor bajo "max_depth"
    # (ya viene en best_params como "max_depth")
    pass

# F1 del mejor trial (best_value)
best_rf_mean_f1 = study.best_value

# Para consistencia de tabla, dejamos std como NaN (no se calculó std del mejor trial)
# Si quisieras std, habría que guardarlo en el objective. Aquí mantenemos la estructura pedida.
models_results["RandomForestClassifier (Optuna optimizado)"] = {
    "mean_f1": best_rf_mean_f1,
    "std_f1": np.nan
}

# Crear tabla comparativa
comparison_df = pd.DataFrame.from_dict(models_results, orient="index")
comparison_df = comparison_df.rename(columns={"mean_f1": "F1_mean_CV", "std_f1": "F1_std_CV"})
comparison_df = comparison_df.sort_values(by="F1_mean_CV", ascending=False)

comparison_df


,F1_mean_CV,F1_std_CV
GradientBoostingClassifier,0.977453,1.021712e-03
LogisticRegression,0.976834,7.909837e-04
RandomForestClassifier (Optuna optimizado),0.975970,NaN
RandomForestClassifier,0.974725,6.358876e-04
DummyClassifier (prior baseline),0.913021,3.493547e-07


### Selección del mejor modelo

De acuerdo con los resultados obtenidos mediante validación cruzada estratificada, el modelo con mejor desempeño fue **GradientBoostingClassifier**, alcanzando un F1 promedio de **0.977453**.

Si bien las diferencias entre los modelos evaluados son reducidas, Gradient Boosting obtuvo el valor más alto de F1, superando ligeramente a Regresión Logística (0.976834) y a Random Forest optimizado mediante Optuna (0.975970).

Además, la baja desviación estándar observada indica que el rendimiento del modelo es consistente entre los distintos folds de validación, lo que sugiere una buena capacidad de generalización.

Por otra parte, todos los modelos supervisados obtuvieron resultados considerablemente superiores al baseline DummyClassifier (F1 = 0.913021), evidenciando que las variables disponibles contienen información predictiva relevante para estimar la variable objetivo `is_recommended`.

En consecuencia, **GradientBoostingClassifier** se selecciona como el mejor modelo de esta etapa y será utilizado como referencia para las siguientes fases del proyecto.

In [10]:
# Selección del mejor modelo según F1_mean_CV
best_model_name = comparison_df.index[0]
best_model_name

'GradientBoostingClassifier'

## Entrenamiento final y persistencia

Se entrenará el **mejor modelo** usando TODO el conjunto de entrenamiento:
- `X_train_processed` (en este notebook: `X_train`)
- `y_train` (en este notebook: `y_train`)

Luego se guardará con Joblib en:
- `models/best_model.pkl`

Si la carpeta `models/` no existiera, se creará automáticamente.


In [11]:
# -------------------------------------------------------
# Creación de carpeta para guardar el modelo
# -------------------------------------------------------

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

best_model_path = models_dir / "best_model.pkl"

# -------------------------------------------------------
# Selección del mejor modelo
# -------------------------------------------------------

if best_model_name == "GradientBoostingClassifier":

    final_model = GradientBoostingClassifier(
        random_state=42
    )

elif best_model_name == "LogisticRegression":

    final_model = LogisticRegression(
        max_iter=1000
    )

elif best_model_name == "RandomForestClassifier":

    final_model = RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )

elif best_model_name == "RandomForestClassifier (Optuna optimizado)":

    final_model = RandomForestClassifier(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        min_samples_split=best_params["min_samples_split"],
        min_samples_leaf=best_params["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )

else:

    final_model = DummyClassifier(
        strategy="prior"
    )

print("Modelo seleccionado:", best_model_name)

# -------------------------------------------------------
# Entrenamiento final usando TODO el train
# -------------------------------------------------------

final_model.fit(
    X_train,
    y_train
)

# -------------------------------------------------------
# Guardado del modelo
# -------------------------------------------------------

joblib.dump(
    final_model,
    best_model_path
)

print(f"Modelo guardado en: {best_model_path.resolve()}")

Modelo seleccionado: GradientBoostingClassifier
Modelo guardado en: C:\Users\maria\OneDrive\Documentos\Informatica\Desarrollo de Aplicaciones Móviles\Proyectos\DataSet-Productos-Cosmeticos - copia\models\best_model.pkl


In [12]:
# Validación rápida (no usa test): verificar que el modelo se entrenó
# y que el artefacto fue guardado correctamente.
from sklearn.base import is_classifier

loaded_model = joblib.load(best_model_path)
print("¿Modelo cargado es clasificador?:", is_classifier(loaded_model))
print("Nombre clase del modelo cargado:", loaded_model.__class__.__name__)


¿Modelo cargado es clasificador?: True
Nombre clase del modelo cargado: GradientBoostingClassifier


## Conclusiones del modelado supervisado

En este notebook se entrenaron y compararon distintos modelos de clasificación supervisada para predecir la variable objetivo **`is_recommended`** utilizando exclusivamente el conjunto de entrenamiento preprocesado.

Los resultados obtenidos mediante **validación cruzada estratificada (StratifiedKFold)** mostraron que todos los modelos evaluados alcanzaron un desempeño elevado, con valores de **F1-score superiores a 0.97**, lo que indica una alta capacidad predictiva sobre los datos disponibles.

El modelo con mejor rendimiento fue **GradientBoostingClassifier**, obteniendo un **F1 promedio de 0.977453**, superando levemente a Regresión Logística y Random Forest. Además, presentó una desviación estándar muy baja entre los folds de validación, evidenciando estabilidad y consistencia en su desempeño.

También se realizó una búsqueda de hiperparámetros mediante **Optuna** para Random Forest. Aunque la optimización mejoró ligeramente el rendimiento del modelo respecto a su configuración por defecto, no logró superar al desempeño obtenido por Gradient Boosting.

Finalmente, el modelo seleccionado fue entrenado utilizando la totalidad del conjunto de entrenamiento (`X_train` y `y_train`) y posteriormente almacenado en el archivo:

`models/best_model.pkl`

Este modelo será utilizado en la siguiente etapa del proyecto para realizar la evaluación final sobre el conjunto de prueba (`X_test`, `y_test`), el cual no ha sido utilizado durante el entrenamiento ni durante la selección del modelo, garantizando así una medición objetiva de la capacidad de generalización.